In [1]:
import numpy as np
import cv2
from PIL import Image
import fitz
import zipfile
import os
import io
import base64

from IPython.display import display, HTML
import ipywidgets as widgets

In [2]:
# ---------- IMAGE ----------
def analyze_image_complexity(image):
    gray = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 100, 200)
    return np.sum(edges) / edges.size


def compress_image_from_bytes(content):
    image = Image.open(io.BytesIO(content)).convert("RGB")
    complexity = analyze_image_complexity(image)

    if complexity > 0.15:
        quality = 70
    elif complexity > 0.08:
        quality = 50
    else:
        quality = 30

    output_path = "compressed.jpg"
    image.save(output_path, "JPEG", optimize=True, quality=quality)

    return output_path, f"Image | Complexity: {complexity:.4f} | Quality: {quality}"


# ---------- PDF ----------
def compress_pdf_from_bytes(content):
    input_path = "temp_input.pdf"
    with open(input_path, "wb") as f:
        f.write(content)

    doc = fitz.open(input_path)

    for page in doc:
        for img in page.get_images(full=True):
            xref = img[0]
            try:
                base = doc.extract_image(xref)
                img_bytes = base["image"]

                img_pil = Image.open(io.BytesIO(img_bytes)).convert("RGB")
                img_pil.save("temp.jpg", "JPEG", quality=40)

                with open("temp.jpg", "rb") as f:
                    doc.update_stream(xref, f.read())
            except:
                pass

    output_path = "compressed.pdf"
    doc.save(output_path, garbage=4, deflate=True)
    doc.close()

    return output_path, "PDF compressed"


# ---------- DOCX ----------
def compress_docx_from_bytes(content):
    input_path = "temp_input.docx"
    with open(input_path, "wb") as f:
        f.write(content)

    with zipfile.ZipFile(input_path, 'r') as zip_ref:
        zip_ref.extractall("temp_docx")

    media_path = "temp_docx/word/media"

    if os.path.exists(media_path):
        for img_file in os.listdir(media_path):
            img_path = os.path.join(media_path, img_file)
            try:
                img = Image.open(img_path).convert("RGB")

                if img.size[0] * img.size[1] > 1000000:
                    quality = 40
                else:
                    quality = 60

                img.save(img_path, optimize=True, quality=quality)
            except:
                pass

    output_path = "compressed.docx"

    with zipfile.ZipFile(output_path, 'w') as zip_out:
        for folder, _, files in os.walk("temp_docx"):
            for file in files:
                path = os.path.join(folder, file)
                zip_out.write(path, os.path.relpath(path, "temp_docx"))

    return output_path, "DOCX compressed"

In [4]:
upload = widgets.FileUpload(
    accept='.jpg,.jpeg,.png,.pdf,.docx',
    multiple=False
)

button = widgets.Button(description="Compress File", button_style='success')
output = widgets.Output()

display(upload, button, output)

def on_click(b):
    with output:
        output.clear_output()

        print("🔍 Checking upload...")

        # -------- CHECK FILE --------
        if not upload.value:
            print("❌ No file uploaded")
            return

        # -------- HANDLE BOTH FORMATS --------
        try:
            # New Jupyter (tuple)
            file = upload.value[0]
            file_name = file['name']
            content = file['content']
        except:
            # Old Jupyter (dict)
            file_name = list(upload.value.keys())[0]
            content = upload.value[file_name]['content']

        print(f"📄 File detected: {file_name}")

        ext = file_name.split(".")[-1].lower()

        # -------- PROCESS --------
        if ext in ["jpg", "jpeg", "png"]:
            output_path, msg = compress_image_from_bytes(content)
            display(Image.open(output_path))

        elif ext == "pdf":
            output_path, msg = compress_pdf_from_bytes(content)

        elif ext == "docx":
            output_path, msg = compress_docx_from_bytes(content)

        else:
            print("❌ Unsupported file type")
            return

        print("✅", msg)

        # -------- DOWNLOAD BUTTON --------
        with open(output_path, "rb") as f:
            data = f.read()

        import base64
        b64 = base64.b64encode(data).decode()

        download_html = f'''
        <a download="{output_path}" href="data:file;base64,{b64}">
            <button style="background-color:green;color:white;padding:10px;border:none;border-radius:5px;">
                ⬇️ Download {output_path}
            </button>
        </a>
        '''

        display(HTML(download_html))

button.on_click(on_click)

FileUpload(value=(), accept='.jpg,.jpeg,.png,.pdf,.docx', description='Upload')

Button(button_style='success', description='Compress File', style=ButtonStyle())

Output()

In [8]:
import numpy as np
import cv2
from PIL import Image
import fitz  # PyMuPDF
import zipfile
import os
import io
import base64
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output  # ← clear_output must be here

In [9]:
# ---------- IMAGE ----------
def analyze_image_complexity(image):
    gray = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 100, 200)
    return np.sum(edges) / edges.size

def compress_image(file_path):
    with open(file_path, "rb") as f:
        content = f.read()
    image = Image.open(io.BytesIO(content)).convert("RGB")

    # Resize down to 50% dimensions (kills 75% of pixels)
    new_size = (int(image.width * 0.5), int(image.height * 0.5))
    image = image.resize(new_size, Image.LANCZOS)

    complexity = analyze_image_complexity(image)

    # Very aggressive quality — targets 80-90% reduction
    quality = 15 if complexity > 0.15 else (12 if complexity > 0.08 else 8)

    output_path = "compressed.jpg"
    image.save(output_path, "JPEG", optimize=True, quality=quality)
    return output_path, f"Image | Complexity: {complexity:.4f} | Quality: {quality} | Resized to 50%"

# ---------- PDF ----------
def compress_pdf(file_path):
    doc = fitz.open(file_path)
    for page in doc:
        for img in page.get_images(full=True):
            xref = img[0]
            try:
                base = doc.extract_image(xref)
                img_pil = Image.open(io.BytesIO(base["image"])).convert("RGB")

                # Resize image to 40% before re-embedding
                new_size = (int(img_pil.width * 0.4), int(img_pil.height * 0.4))
                img_pil = img_pil.resize(new_size, Image.LANCZOS)

                buf = io.BytesIO()
                img_pil.save(buf, "JPEG", quality=15)
                doc.update_stream(xref, buf.getvalue())
            except:
                pass

    output_path = "compressed.pdf"
    # garbage=4 removes unused objects, deflate=True max-compresses streams
    doc.save(output_path, garbage=4, deflate=True, clean=True)
    doc.close()
    return output_path, "PDF compressed aggressively"

# ---------- DOCX ----------
def compress_docx(file_path):
    extract_dir = "temp_docx"
    if os.path.exists(extract_dir):
        import shutil
        shutil.rmtree(extract_dir)

    with zipfile.ZipFile(file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

    media_path = os.path.join(extract_dir, "word", "media")
    if os.path.exists(media_path):
        for img_file in os.listdir(media_path):
            img_path = os.path.join(media_path, img_file)
            try:
                img = Image.open(img_path).convert("RGB")

                # Resize to 40% + very low quality
                new_size = (int(img.width * 0.4), int(img.height * 0.4))
                img = img.resize(new_size, Image.LANCZOS)

                img.save(img_path, "JPEG", optimize=True, quality=15)
            except:
                pass

    output_path = "compressed.docx"
    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=9) as zip_out:
        for folder, _, files in os.walk(extract_dir):
            for file in files:
                path = os.path.join(folder, file)
                zip_out.write(path, os.path.relpath(path, extract_dir))

    return output_path, "DOCX compressed aggressively"

In [11]:
# -------- UPLOAD WIDGET --------
uploader = widgets.FileUpload(
    accept='.jpg,.jpeg,.png,.pdf,.docx',
    multiple=False,
    description='📁 Pick File'
)
run_btn = widgets.Button(
    description="🚀 Compress Now",
    button_style='success',
    layout=widgets.Layout(margin='10px 0')
)
output_area = widgets.Output()

def on_compress(b):
    with output_area:
        clear_output()

        if not uploader.value:
            print("❌ No file uploaded yet. Please click 'Pick File' first.")
            return

        try:
            if isinstance(uploader.value, tuple):
                file_info = uploader.value[0]
                filename  = file_info['name']
                content   = bytes(file_info['content'])
            else:
                filename = list(uploader.value.keys())[0]
                content  = bytes(uploader.value[filename]['content'])
        except Exception as e:
            print(f"❌ Could not read uploaded file: {e}")
            print(f"   Debug — type: {type(uploader.value)}")
            print(f"   Debug — value: {uploader.value}")
            return

        # ---- SMART FILENAME: xyz.pdf → xyz_compressed.pdf ----
        base_name, ext_with_dot = os.path.splitext(filename)   # "xyz", ".pdf"
        ext = ext_with_dot.lstrip(".").lower()                  # "pdf"
        download_filename = f"{base_name}_compressed{ext_with_dot}"  # "xyz_compressed.pdf"

        orig_size = len(content) / 1024
        print(f"📂 File     : {filename}")
        print(f"📏 Size     : {orig_size:.1f} KB")
        print(f"🔍 Type     : {ext.upper()}")
        print(f"💾 Will save as : {download_filename}")
        print("⏳ Compressing...")

        temp_input = f"temp_input.{ext}"
        with open(temp_input, "wb") as f:
            f.write(content)

        try:
            if ext in ["jpg", "jpeg", "png"]:
                output_path, msg = compress_image(temp_input)
            elif ext == "pdf":
                output_path, msg = compress_pdf(temp_input)
            elif ext == "docx":
                output_path, msg = compress_docx(temp_input)
            else:
                print(f"❌ Unsupported: .{ext} — use JPG, PNG, PDF or DOCX")
                return
        except Exception as e:
            print(f"❌ Compression failed: {e}")
            import traceback; traceback.print_exc()
            return

        # -------- STATS --------
        comp_size = os.path.getsize(output_path) / 1024
        saved     = orig_size - comp_size
        pct       = (saved / orig_size * 100) if orig_size > 0 else 0

        # Color code the result
        color = "#28a745" if pct >= 70 else ("#ffc107" if pct >= 40 else "#dc3545")
        print(f"\n✅ {msg}")
        print(f"{'─'*45}")
        print(f"📦 Original   : {orig_size:.1f} KB")
        print(f"📉 Compressed : {comp_size:.1f} KB")
        print(f"💾 Saved      : {saved:.1f} KB")
        display(HTML(f'<b style="color:{color};font-size:16px">🎯 {pct:.1f}% size reduction</b>'))
        print(f"{'─'*45}")

        # Image preview
        if ext in ["jpg", "jpeg", "png"]:
            print("\n🖼️ Preview:")
            display(Image.open(output_path))

        # -------- DOWNLOAD BUTTON with smart name --------
        with open(output_path, "rb") as f:
            data = f.read()
        b64 = base64.b64encode(data).decode()

        display(HTML(f"""
        <br>
        <a download="{download_filename}" href="data:application/octet-stream;base64,{b64}">
            <button style="
                background:#28a745;color:white;padding:12px 28px;
                border:none;border-radius:6px;font-size:15px;cursor:pointer;
                box-shadow:0 2px 6px rgba(0,0,0,0.2)">
                ⬇️ Download — {download_filename}
            </button>
        </a>
        """))

run_btn.on_click(on_compress)

display(widgets.VBox([
    widgets.HTML("<h3>🗜️ File Compressor</h3>"),
    widgets.HTML("<p>Step 1 → Upload file &nbsp;|&nbsp; Step 2 → Click Compress</p>"),
    uploader,
    run_btn,
    output_area
]))